# Reproducing *The Prompt Proposes, the Code Decides*

This notebook regenerates the paper's tables from the saved per-task and per-call results in
[github.com/RobertoVetere/nervo-gemma-paper](https://github.com/RobertoVetere/nervo-gemma-paper). It needs no GPU,
no model and no API key (internet on, to clone the repository). Each section prints the numbers next to where they
appear in the paper. Re-running the experiments themselves is described in `docs/REPRODUCE.md`.

In [ ]:
import os, subprocess
if not os.path.isdir('nervo-gemma-paper'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RobertoVetere/nervo-gemma-paper'], check=True)
os.chdir('nervo-gemma-paper')
def run(cmd):
    print('$', cmd); print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

## 4.2 A controlled test of the call format

80 blocks of real code × 2 samples × 3 interfaces, E4B with the official 4-bit weights (`modal/e4b`) and the original
bf16 weights (`modal/e4b-bf16`) on the same vLLM server and Gemma 4 tool parser, with and without ~10,000 tokens of
previously read code (1,920 calls). Rows `deepinfra/e4b` were requested as E4B from an API that served a 31B (paper,
4.6). The column *Fields lost / unparsable* is the paper's table.

In [ ]:
run('python3 -I lab/oneshot/resumen_formato.py resultados/formato_llamadas/e4b_*.jsonl')

## 4.3 Reasoning × agent structure, three replicas

Same 40 random validation tasks (`local/muestra.py --semilla 10 --n 40`), one run per cell per replica, through the
API that served the 31B (paper, 4.6). Cells: free-form agent with and without reasoning (`base_comunidad`, `e4b_t0`), `fixer` with and without
reasoning (`e4b_cmd_t`, `e4b_cmd`). Interaction per task = (free-form gain from reasoning) − (fixer gain).

In [ ]:
run('python3 -I local/tabla_replicas.py resultados/local_deepinfra/l40 resultados/local_deepinfra/l40_r2 resultados/local_deepinfra/l40_r3')
run('python3 -I local/interaccion.py')
run('python3 -I local/tabla_replicas.py resultados/local_deepinfra/l40_r2 resultados/local_deepinfra/l40_r3 | tail -4')  # new replicas alone

## 4.3 First replica in detail (McNemar tests, Wilson intervals, turn-limit cut-offs)

In [ ]:
run('python3 -I local/tabla_lote.py resultados/local_deepinfra/l40 | head -40')

## 4.4–4.5 Single-call probes: E4B bf16 vs 4-bit on one server, 31B (API), whole-function rewrite

In [ ]:
run('python3 -I lab/oneshot/resumen_cuantizacion.py')

## 4.1 Tool-call failures in agent runs

One row per evaluation run, with its run id; the paper's table sums rows per interface.

In [ ]:
run('cat resultados/formato/RESUMEN.md')